# ΔΔG generalization benchmark

Runs the holdout suite (`ddg.evaluation`) over a `features_summary.parquet` and
shows the summary table + figures inline. See `docs/benchmark_plan.md` for what
each holdout means. Edit `EXPERIMENT` / `MODEL` below.

Every holdout is a train/test split of the *same* feature table — Boltz is not
re-run here.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from pathlib import Path
from IPython.display import Image, display

from ddg.evaluation.benchmark import run_benchmark
from ddg.evaluation import plots as P
from ddg.evaluation import cluster as C

EXPERIMENT = 'tsuboyama_bench_fast'   # or tsuboyama_bench_wide
MODEL = 'svr'                          # svr | ridge | mlp
PROC = Path('..') / 'data' / 'processed' / EXPERIMENT
OUT = PROC / 'benchmark'
df = pd.read_parquet(PROC / 'features_summary.parquet')
print(df.shape, '|', df.wt_id.nunique(), 'proteins')

## (optional) homology clusters for the cluster holdout
Needs the `mmseqs` binary. Skip this cell to run every holdout except `cluster`.

In [ ]:
cluster_map = None
try:
    cluster_map = C.cluster_wt_sequences(PROC / 'wt_sequences.fasta',
                                         min_seq_id=0.3, out_csv=OUT / 'cluster_map.csv')
    print('clusters:', len(set(cluster_map.values())))
except Exception as e:
    print('cluster holdout will be skipped:', e)

In [ ]:
results = run_benchmark(df, model_name=MODEL, out_dir=OUT, cluster_map=cluster_map)
results.summary

## Figures

In [ ]:
figs = P.make_all(results, OUT / 'figures')
for f in figs:
    display(Image(str(f)))

## Per-unit detail (the distribution that a mean hides)
Worst proteins / clusters — where does the model fail?

In [ ]:
for h in ['protein', 'cluster', 'chemistry']:
    if h in results.per_unit:
        print(f'\n=== {h}: worst 10 by Pearson ===')
        display(results.per_unit[h].dropna(subset=['pearson']).sort_values('pearson').head(10))